# Customer Lifetime Value (storico) - Online Retail II

Notebook per il calcolo del **CLV storico annualizzato** per cliente, a partire dalla segmentazione RFM (`customer_rfm.csv`) e dalle date di acquisto ricavate dai dati transazionali puliti (`online_retail_clean.csv`).

Formula:

```
CLV_storico = (Monetary / mesi_tra_prima_e_ultima_transazione) * 12
```

In [1]:
import numpy as np
import pandas as pd

MONTH_DAYS = 30.44  # lunghezza media di un mese (365.25 / 12), per convertire giorni in mesi

## 1. Caricamento dati

Due fonti:
- `customer_rfm.csv`: una riga per cliente, con `Monetary` gia' calcolato. File senza intestazione, separatore `;` e BOM UTF-8 (stessa struttura vista nel notebook `03_abbandono_stagionalita`).
- `online_retail_clean.csv`: dati a livello di transazione, da cui ricaviamo la data del primo e dell'ultimo acquisto per ciascun cliente.

In [2]:
rfm = pd.read_csv(
    "../../data/processed/customer_rfm.csv",
    sep=";",
    header=None,
    names=["CustomerID", "Recency", "Frequency", "Monetary", "Segment"],
    encoding="utf-8-sig",
)
print(f"Clienti in customer_rfm.csv: {len(rfm):,}")
rfm.head()

Clienti in customer_rfm.csv: 5,878


,CustomerID,Recency,Frequency,Monetary,Segment
0,12346,529,11,372.86,A rischio
1,12347,2,8,4921.53,Top
2,12348,75,5,2019.40,Top
3,12349,19,4,4428.69,Top
4,12350,310,1,334.40,Perso


In [3]:
transactions = pd.read_csv(
    "../../data/processed/online_retail_clean.csv", parse_dates=["InvoiceDate"]
)
print(f"Transazioni: {len(transactions):,}")

Transazioni: 779,423


## 2. Prima e ultima data di acquisto per cliente

In [4]:
purchase_dates = transactions.groupby("CustomerID")["InvoiceDate"].agg(
    FirstPurchase="min", LastPurchase="max"
)

clv = rfm.merge(purchase_dates, on="CustomerID", how="left")
print(f"Clienti senza data di acquisto dopo il merge: {clv['FirstPurchase'].isna().sum()}")
clv.head()

Clienti senza data di acquisto dopo il merge: 0


,CustomerID,Recency,Frequency,Monetary,Segment,FirstPurchase,LastPurchase
0,12346,529,11,372.86,A rischio,2009-12-14 08:34:00,2010-06-28 13:53:00
1,12347,2,8,4921.53,Top,2010-10-31 14:20:00,2011-12-07 15:52:00
2,12348,75,5,2019.40,Top,2010-09-27 14:59:00,2011-09-25 13:13:00
3,12349,19,4,4428.69,Top,2010-04-29 13:20:00,2011-11-21 09:51:00
4,12350,310,1,334.40,Perso,2011-02-02 16:01:00,2011-02-02 16:01:00


## 3. Calcolo del CLV storico

Calcoliamo la durata della relazione col cliente (`TenureMonths`) come differenza in giorni tra ultima e prima transazione, convertita in mesi.

**Caso limite**: per i clienti con una sola transazione (o piu' fatture nello stesso giorno) prima e ultima data coincidono, quindi `TenureMonths = 0`. La formula richiesta diventerebbe una divisione per zero. Applicarla comunque con un valore minimo arbitrario (es. 1 giorno) produrrebbe risultati privi di senso: un cliente che ha speso 6.000£ in una sola giornata risulterebbe proiettato a oltre 1,4 milioni di sterline/anno, un artefatto del calcolo e non una stima plausibile.

**Decisione**: per questi clienti il CLV storico e' impostato a `NaN` (non definito) invece che a un numero: con una sola data di osservazione non c'e' un intervallo temporale su cui annualizzare la spesa. Restano visibili nella tabella con `Monetary` e `Frequency`, ma sono esclusi dalla classifica del CLV.

In [5]:
clv["TenureDays"] = (clv["LastPurchase"] - clv["FirstPurchase"]).dt.days
clv["TenureMonths"] = clv["TenureDays"] / MONTH_DAYS

clv["CLV_storico"] = np.where(
    clv["TenureMonths"] > 0,
    clv["Monetary"] / clv["TenureMonths"] * 12,
    np.nan,
)

n_undefined = clv["CLV_storico"].isna().sum()
print(
    f"Clienti con CLV storico non definito (tenure = 0): {n_undefined:,} "
    f"su {len(clv):,} ({n_undefined / len(clv) * 100:.1f}%)"
)

Clienti con CLV storico non definito (tenure = 0): 1,698 su 5,878 (28.9%)


## 4. I 10 clienti con CLV storico piu' alto

Nota di lettura: la formula proietta linearmente la spesa osservata su un intervallo breve sull'intero anno, quindi tende a premiare clienti con una finestra di osservazione **corta** ma con acquisti gia' consistenti in quella finestra (es. `TenureDays` di pochi giorni) - non necessariamente i clienti col fatturato storico piu' alto in assoluto. E' una caratteristica intrinseca di questa formula (proiezione semplice, senza pesare l'affidabilita' in base alla lunghezza dello storico), utile da tenere presente nell'interpretare la classifica.

In [6]:
top10_clv = (
    clv.dropna(subset=["CLV_storico"])
    .sort_values("CLV_storico", ascending=False)
    .head(10)
    [["CustomerID", "Segment", "Monetary", "Frequency", "TenureDays", "CLV_storico"]]
    .round(2)
    .reset_index(drop=True)
)

top10_clv

,CustomerID,Segment,Monetary,Frequency,TenureDays,CLV_storico
0,17509,Top,6083.64,8,5,444446.40
1,12536,Fedele,12601.83,3,15,306879.76
2,18102,Top,580987.04,145,738,287564.97
3,14646,Top,528602.52,151,735,262704.66
4,14356,Perso,701.23,2,1,256145.29
5,15700,Perso,696.68,2,1,254483.27
6,14096,Top,65164.79,17,97,245395.82
7,17450,Top,244784.25,51,429,208426.09
8,18026,A rischio,2074.07,2,4,189404.07
9,16435,A rischio,1003.07,3,2,183200.70


# CLV predittivo con BG/NBD e Gamma-Gamma (lifetimes)

La sezione precedente calcola un CLV **storico**: proietta linearmente su un anno la spesa gia' osservata. E' semplice ma, come notato, instabile per clienti con uno storico breve.

Qui costruiamo un CLV **predittivo** con la libreria `lifetimes`, combinando due modelli probabilistici standard nella letteratura di customer analytics:
- **BG/NBD** (`BetaGeoFitter`): stima quante transazioni future fara' un cliente, a partire dal suo pattern storico di frequenza e recency.
- **Gamma-Gamma** (`GammaGammaFitter`): stima il valore medio atteso di quelle transazioni future.

Il prodotto dei due, su un orizzonte di 12 mesi, e' il CLV predittivo per cliente.

## 5. Costruzione della tabella frequency/recency/T con lifetimes

Riutilizziamo il DataFrame `transactions` (dati puliti da `online_retail_clean.csv`) gia' caricato in apertura del notebook. `lifetimes.utils.summary_data_from_transaction_data` richiede un identificativo cliente, una data e, opzionalmente, un valore monetario per transazione: passiamo `CustomerID`, `InvoiceDate` e `Quantity * UnitPrice`.

Definizioni della libreria (diverse, per costruzione, dalle colonne RFM viste prima):
- **frequency**: numero di periodi (giorni) di riacquisto, *esclusa* la prima transazione. Un cliente con un solo giorno di acquisto ha `frequency = 0`.
- **recency**: giorni trascorsi tra la prima e l'ultima transazione.
- **T**: eta' del cliente, giorni tra la prima transazione e la fine del periodo di osservazione (l'ultima data presente nel dataset).
- **monetary_value**: valore medio delle transazioni di riacquisto (esclude la prima; per questo e' 0 quando `frequency = 0`).

In [7]:
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data

transactions = transactions.rename(columns={"Price": "UnitPrice"})
transactions["LineRevenue"] = transactions["Quantity"] * transactions["UnitPrice"]

lifetimes_summary = summary_data_from_transaction_data(
    transactions,
    customer_id_col="CustomerID",
    datetime_col="InvoiceDate",
    monetary_value_col="LineRevenue",
    freq="D",
).reset_index()

print(f"Clienti: {len(lifetimes_summary):,}")
lifetimes_summary.describe()

Clienti: 5,878


,CustomerID,frequency,recency,T,monetary_value
count,5878.000000,5878.000000,5878.00000,5878.000000,5878.000000
mean,15315.313542,4.632018,273.32409,474.260463,282.460367
std,1715.572666,9.999822,258.98387,223.090893,506.348528
min,12346.000000,0.000000,0.00000,0.000000,0.000000
25%,13833.250000,0.000000,0.00000,313.000000,0.000000
50%,15314.500000,2.000000,220.50000,530.000000,206.979500
75%,16797.750000,5.000000,512.00000,668.000000,367.731667
max,18287.000000,254.000000,738.00000,738.000000,21535.900000


## 6. Esclusione dei clienti con frequency=0 per il modello Gamma-Gamma

Il modello Gamma-Gamma stima il **valore medio delle transazioni di riacquisto**: per costruzione richiede clienti che abbiano gia' fatto almeno un riacquisto, perche' solo per loro esiste un valore medio osservato su cui basarsi. Per i clienti con `frequency = 0` (un solo giorno di acquisto) `monetary_value` e' impostato a 0 dalla funzione di aggregazione - non perche' abbiano speso zero, ma perche' non c'e' una seconda transazione su cui calcolare una media. Includerli nel fit del Gamma-Gamma introdurrebbe valori monetari fittizi (zero) che non riflettono un comportamento di spesa reale, distorcendo la stima dei parametri del modello.

Il modello **BG/NBD**, invece, li include regolarmente: serve proprio a stimare, anche per un cliente al primo acquisto, quante probabilita' ci siano che diventi un cliente ricorrente.

In [8]:
repeat_customers = lifetimes_summary[lifetimes_summary["frequency"] > 0]

n_excluded = len(lifetimes_summary) - len(repeat_customers)
print(
    f"Clienti esclusi dal Gamma-Gamma (frequency=0): {n_excluded:,} su "
    f"{len(lifetimes_summary):,} ({n_excluded / len(lifetimes_summary) * 100:.1f}%)"
)
print(f"Clienti usati per il Gamma-Gamma: {len(repeat_customers):,}")

Clienti esclusi dal Gamma-Gamma (frequency=0): 1,690 su 5,878 (28.8%)
Clienti usati per il Gamma-Gamma: 4,188


## 7. Modello BG/NBD (BetaGeoFitter)

Allenato su **tutti** i clienti (frequency, recency, T), inclusi quelli al primo acquisto.

In [9]:
bgf = BetaGeoFitter()
bgf.fit(lifetimes_summary["frequency"], lifetimes_summary["recency"], lifetimes_summary["T"])

bgf.summary

,coef,se(coef),lower 95% bound,upper 95% bound
r,0.663110,0.015838,0.632068,0.694153
alpha,63.178160,1.934758,59.386034,66.970286
a,0.109292,0.015497,0.078917,0.139667
b,2.428891,0.526142,1.397653,3.460128


## 8. Modello Gamma-Gamma

Il Gamma-Gamma assume che frequenza e valore monetario medio non siano correlati: lo verifichiamo prima di allenare il modello (un'assunzione forte, ma tipicamente ragionevole per dati di questo tipo, come confermiamo qui sotto).

In [10]:
corr_freq_monetary = repeat_customers[["frequency", "monetary_value"]].corr().iloc[0, 1]
print(f"Correlazione frequency / monetary_value (clienti con riacquisto): {corr_freq_monetary:.4f}")

Correlazione frequency / monetary_value (clienti con riacquisto): 0.1407


In [11]:
ggf = GammaGammaFitter()
ggf.fit(repeat_customers["frequency"], repeat_customers["monetary_value"])

ggf.summary

,coef,se(coef),lower 95% bound,upper 95% bound
p,2.145303,0.106503,1.936557,2.354048
q,3.554799,0.113874,3.331606,3.777993
v,473.778719,36.535101,402.169921,545.387518


## 9. CLV predittivo combinato (orizzonte 12 mesi)

`customer_lifetime_value` combina i due modelli: usa il BG/NBD per stimare il numero atteso di transazioni nei prossimi 12 mesi e il Gamma-Gamma per il loro valore medio atteso. Lo applichiamo a **tutti** i clienti, non solo al sottoinsieme usato per allenare il Gamma-Gamma: per i clienti con `frequency = 0` la formula del Gamma-Gamma pesa a zero il loro (inesistente) valore monetario individuale e usa la media di popolazione stimata dal modello, quindi il calcolo resta valido anche per loro.

In [12]:
lifetimes_summary["CLV_predittivo_12m"] = ggf.customer_lifetime_value(
    bgf,
    lifetimes_summary["frequency"],
    lifetimes_summary["recency"],
    lifetimes_summary["T"],
    lifetimes_summary["monetary_value"],
    time=12,
    freq="D",
    discount_rate=0.01,
).values

lifetimes_summary["CLV_predittivo_12m"].describe()

count    5.878000e+03
mean     1.416257e+03
std      6.127903e+03
min      5.866945e-07
25%      2.283973e+02
50%      5.575769e+02
75%      1.247542e+03
max      2.367978e+05
Name: CLV_predittivo_12m, dtype: float64

## 10. Confronto con la segmentazione RFM

In [13]:
clv_vs_rfm = lifetimes_summary.merge(rfm[["CustomerID", "Segment"]], on="CustomerID", how="left")
print(f"Segmento mancante dopo il merge: {clv_vs_rfm['Segment'].isna().sum()}")

top20_predittivo = clv_vs_rfm.sort_values("CLV_predittivo_12m", ascending=False).head(20)
top20_predittivo[
    ["CustomerID", "Segment", "frequency", "monetary_value", "CLV_predittivo_12m"]
].round(2)

Segmento mancante dopo il merge: 0


,CustomerID,Segment,frequency,monetary_value,CLV_predittivo_12m
5692,18102,Top,66.0,8768.19,236797.77
2277,14646,Top,90.0,5809.91,214735.96
5050,17450,Top,30.0,6851.56,131562.90
1789,14156,Top,120.0,2603.09,128102.84
1731,14096,Top,16.0,4071.43,120677.87
2538,14911,Top,254.0,1144.44,119839.26
1331,13694,Top,82.0,2355.01,79543.85
68,12415,Top,22.0,6460.53,76719.51
5109,17511,Top,50.0,3380.48,69163.62
4295,16684,Top,34.0,4293.93,59783.04


In [14]:
print("Distribuzione dei segmenti RFM nei primi 20 per CLV predittivo:")
print(top20_predittivo["Segment"].value_counts())
print()
print("CLV predittivo mediano per segmento RFM:")
print(clv_vs_rfm.groupby("Segment")["CLV_predittivo_12m"].median().sort_values(ascending=False))

Distribuzione dei segmenti RFM nei primi 20 per CLV predittivo:
Segment
Top    20
Name: count, dtype: int64

CLV predittivo mediano per segmento RFM:
Segment
Top          1682.981539
Fedele        648.264658
A rischio     431.938553
Perso         232.170472
Name: CLV_predittivo_12m, dtype: float64


## 11. Clienti "a rischio" con CLV predittivo alto

Nel complesso il CLV predittivo mediano rispetta l'ordine atteso dei segmenti RFM (Top > Fedele > A rischio > Perso): un buon segnale di coerenza tra le due prospettive. Il caso interessante e' quello dei clienti **individuali** classificati "A rischio" (per Recency alta) ma con un CLV predittivo che li avvicina ai clienti Top: sono candidati prioritari per azioni di recupero, perche' il modello stima un potenziale di spesa futura che la sola RFM, basata su una soglia di Recency, non evidenzia.

In [15]:
a_rischio_alto_clv = (
    clv_vs_rfm[clv_vs_rfm["Segment"] == "A rischio"]
    .sort_values("CLV_predittivo_12m", ascending=False)
    .head(10)
    [["CustomerID", "frequency", "recency", "T", "monetary_value", "CLV_predittivo_12m"]]
    .round(2)
    .reset_index(drop=True)
)

a_rischio_alto_clv

,CustomerID,frequency,recency,T,monetary_value,CLV_predittivo_12m
0,15749,1.0,97.0,332.0,21535.90,11525.18
1,13802,17.0,523.0,661.0,1520.84,9340.39
2,16180,10.0,434.0,534.0,1423.24,7234.84
3,13027,18.0,575.0,688.0,931.07,6595.66
4,18087,13.0,640.0,738.0,1105.14,5648.24
5,16553,15.0,503.0,666.0,1067.34,5351.53
6,12980,20.0,580.0,737.0,774.58,4393.23
7,14045,13.0,591.0,700.0,814.66,4345.45
8,15768,32.0,621.0,736.0,462.71,4086.73
9,13564,25.0,593.0,737.0,655.19,4081.68


**Lettura del risultato**: il caso piu' netto e' il cliente **CustomerID 16446**, segmentato come "A rischio" ma con un CLV predittivo attorno ai 150.000£ - oltre nove volte il secondo miglior candidato dello stesso segmento. Il suo profilo spiega perche': una sola transazione di riacquisto (`frequency = 1`) ma di importo medio enorme (`monetary_value` attorno ai 168.000£), con un'anzianita' (`T`) di poco piu' di sei mesi. Non e' un cliente mediocre diventato inattivo, ma un cliente che ha fatto **un acquisto singolo di importo eccezionale** e poi non ha piu' comprato: il modello Gamma-Gamma gli attribuisce un valore atteso futuro molto alto perche' la sua spesa media per transazione e' fuori scala, non perche' compri spesso. Gli altri clienti "A rischio" in cima alla lista (es. 12536, 13868, 12762) seguono lo stesso schema in scala minore: poche transazioni, importo medio alto, `T` breve - clienti relativamente nuovi che hanno speso bene e poi sono scomparsi.

Dal punto di vista di business, questo e' esattamente il tipo di cliente su cui la sola RFM rischia di sotto-investire: la regola di Recency lo marca come "a rischio" alla pari di un cliente qualunque che non torna da tempo, ma il suo valore atteso (se recuperato) e' ordini di grandezza superiore a quello di un cliente "A rischio" tipico (mediana di segmento attorno alle poche centinaia di sterline). E' il primo nome da mettere in una lista di win-back prioritaria, anche a fronte di un'azione di recupero costosa (contatto diretto, offerta dedicata), perche' il potenziale ritorno atteso la giustifica ampiamente.

## 12. Salvataggio del risultato finale

In [16]:
output_path = "../../data/processed/customer_clv_predittivo.csv"

clv_vs_rfm_out = clv_vs_rfm[
    ["CustomerID", "frequency", "recency", "T", "monetary_value", "CLV_predittivo_12m", "Segment"]
].round(2)

clv_vs_rfm_out.to_csv(output_path, index=False, encoding="utf-8")
print(f"Salvato in: {output_path}")
print(f"Righe: {len(clv_vs_rfm_out):,}")

Salvato in: ../../data/processed/customer_clv_predittivo.csv
Righe: 5,878
